# gemma4:e4b — what happened, document by document

This notebook is written for a reader with no background in machine learning.
It answers one question, ten times over: **for each of the 10 sample documents,
how well did the model do, and exactly what did it get wrong?**

No scores like "F1" or "precision" here — just plain counts ("27 out of 28
correct") and, for anything that wasn't correct, the actual text involved, so
you can judge for yourself whether the mistake matters.


## How grading works, in plain terms

Every document has a human-written **answer key**: someone read the PDF and
wrote down what each piece of text *is* — a title, a section heading, a
question, or an answer. We then ran the same document through the model and
compared its output to that answer key.

A piece of the model's output **counts as correct** when two things both hold:

1. it covers essentially the same words as one answer-key entry, and
2. it was given the same label (title / section heading / question / answer).

Three things can go wrong instead:

| Problem | What it means |
|---|---|
| **Missed** | The answer key has an entry the model never produced anything for |
| **Wrong label** | The model found the right text, but called it the wrong thing |
| **Extra / spurious** | The model produced something that isn't in the answer key at all |

### Two answer keys, not one

We actually check twice, against two *separate*, independently hand-written
answer keys:

- **Path A** checks the model's own output directly, against the original
  answer key.
- **Path B** first runs a small automatic fix-up step (filling in a question
  that was left blank, using the section heading above it) and checks the
  *result* against a second, separately revised answer key.

Because the two paths use different answer keys, a document can score
differently on each — that's expected, not a bug.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import pandas as pd

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import (
    resolve_old_gt_path, extract_gold, _match_structured,
)
from dmpbridge.evaluation.annotation_rules import (
    resolve_new_gt_path, convert_tag_to_final,
)

MODEL, TAG = 'gemma4:e4b', 'gemma4-e4b_pdfplumber_whole_doc'
SAMPLES = range(1, 11)

# Stage 4 (Path B's input) is derived from stage 3 — build it if missing.
if not P.final_path(TAG, 1).exists():
    convert_tag_to_final(TAG)

pd.set_option('display.max_colwidth', 90)


def sample_report(n, path):
    """Return (correct, total, problems_df) for one sample, one path.

    path='A' checks stage 3 against the original answer key with question/
    section-title duplicates removed (the standard Path A setup); path='B'
    checks stage 4 against the revised answer key with nothing removed —
    Path B's own standard setup, since it exists to measure the fill-in step.
    """
    if path == 'A':
        gt = resolve_old_gt_path(n)
        dedup = True
        pred_path = P.structured_path(TAG, n)
    else:
        gt = resolve_new_gt_path(n)
        dedup = False
        pred_path = P.final_path(TAG, n)

    gold = extract_gold(gt, dedup_question_title=dedup)
    records, no_gold = _match_structured(pred_path, gold, dedup_question_title=dedup)

    correct = sum(1 for r in records
                  if r['pred_text'] is not None and r['pred_label'] == r['gold_label'])
    total = len(records) + len(no_gold)

    rows = []
    for r in records:
        if r['pred_text'] is None:
            rows.append({'problem': 'missed', 'label': r['gold_label'],
                         'answer key says': r['gold_text'][:90],
                         'model said': '(nothing)'})
        elif r['pred_label'] != r['gold_label']:
            rows.append({'problem': 'wrong label', 'label': f"{r['gold_label']} \u2192 {r['pred_label']}",
                         'answer key says': r['gold_text'][:90],
                         'model said': r['pred_text'][:90]})
    for text, label in no_gold:
        rows.append({'problem': 'extra / spurious', 'label': label,
                     'answer key says': '(not in the answer key)',
                     'model said': text[:90]})

    problems = pd.DataFrame(rows, columns=['problem', 'label', 'answer key says', 'model said'])
    return correct, total, problems


## Sample by sample

For each document: how many items matched out of how many, for both paths, and
— if anything didn't match — exactly what and why. A document with no table
under it had nothing wrong: every item matched.


In [2]:
PROBLEM_COLOUR = {
    'missed':            '#fdecea',
    'wrong label':       '#fff4e5',
    'extra / spurious':  '#eaf1fb',
}

def show_problems(df):
    if df.empty:
        print('  Nothing to report \u2014 every item matched.')
        return
    def row_colour(row):
        c = PROBLEM_COLOUR.get(row['problem'], '#ffffff')
        return [f'background-color: {c}'] * len(row)
    display(df.style.apply(row_colour, axis=1).hide(axis='index'))

for n in SAMPLES:
    ca, ta, probs_a = sample_report(n, 'A')
    cb, tb, probs_b = sample_report(n, 'B')
    print()
    print('=' * 78)
    print(f'SAMPLE {n}    Path A: {ca}/{ta} correct ({ca/ta:.0%})'
          f'    Path B: {cb}/{tb} correct ({cb/tb:.0%})')
    print('=' * 78)
    print('Path A:')
    show_problems(probs_a)
    print('Path B:')
    show_problems(probs_b)



SAMPLE 1    Path A: 27/31 correct (87%)    Path B: 30/34 correct (88%)
Path A:


problem,label,answer key says,model said
missed,answer.text,"This secondary data analysis project will analyze deidentified data from 48,218 participan",(nothing)
extra / spurious,answer.text,(not in the answer key),"This secondary data analysis project will analyze deidentified data from 48,218 participan"
extra / spurious,question.text,(not in the answer key),about using their data
extra / spurious,answer.text,(not in the answer key),(i) the SOL-VIDA Study – Data requests may be made by following instructions on the Hispan


Path B:


problem,label,answer key says,model said
missed,answer.text,"This secondary data analysis project will analyze deidentified data from 48,218 participan",(nothing)
extra / spurious,answer.text,(not in the answer key),"This secondary data analysis project will analyze deidentified data from 48,218 participan"
extra / spurious,question.text,(not in the answer key),about using their data
extra / spurious,answer.text,(not in the answer key),(i) the SOL-VIDA Study – Data requests may be made by following instructions on the Hispan



SAMPLE 2    Path A: 18/28 correct (64%)    Path B: 20/32 correct (62%)
Path A:


...

Path B:


problem,label,answer key says,model said
missed,section.title,Types of data,(nothing)
missed,question.text,Types of data,(nothing)
missed,answer.text,"The bulk of the data generated in this project will be 1, 2, 3, and 4 dimensional arrays o",(nothing)
missed,section.title,Data and metadata standards,(nothing)
missed,question.text,Data and metadata standards,(nothing)
missed,answer.text,The PI's research group will adopt the longstanding practice that all generated data shoul,(nothing)
missed,section.title,Policies for access and sharing,(nothing)
missed,question.text,Policies for access and sharing,(nothing)
missed,answer.text,Interested parties will be able to request data and codes at the end of the project. Such,(nothing)
missed,section.title,"Policies and provisions for re-use, re-distribution",(nothing)



SAMPLE 7    Path A: 2/2 correct (100%)    Path B: 2/4 correct (50%)
Path A:
  Nothing to report — every item matched.
Path B:


problem,label,answer key says,model said
wrong label,question.text → section.title,Resource/Data Sharing Plan,Resource/Data Sharing Plan
extra / spurious,question.text,(not in the answer key),Resource/Data Sharing Plan



SAMPLE 8    Path A: 13/13 correct (100%)    Path B: 19/19 correct (100%)
Path A:
  Nothing to report — every item matched.
Path B:
  Nothing to report — every item matched.

SAMPLE 9    Path A: 11/11 correct (100%)    Path B: 16/16 correct (100%)
Path A:
  Nothing to report — every item matched.
Path B:
  Nothing to report — every item matched.

SAMPLE 10    Path A: 13/13 correct (100%)    Path B: 19/19 correct (100%)
Path A:
  Nothing to report — every item matched.
Path B:
  Nothing to report — every item matched.


## From ten documents to one big picture

Everywhere else in this project you'll see a single **confusion matrix** —
one grid covering all 10 documents at once, per path. That grid is built from
exactly the counts above: every "correct" tally above lands on the matrix's
diagonal; every "missed" and "wrong label" lands somewhere off it.

The table below makes that connection explicit: it's the same 10 rows you just
read through, side by side, so you can see how much each document contributes
to the final total — and confirm that one unusually bad document (if there is
one) is not being hidden by the average.


In [3]:
rows = []
for n in SAMPLES:
    ca, ta, _ = sample_report(n, 'A')
    cb, tb, _ = sample_report(n, 'B')
    rows.append({'sample': f'sample{n}', 'Path A correct': ca, 'Path A total': ta,
                 'Path A %': f'{ca/ta:.0%}' if ta else '\u2013',
                 'Path B correct': cb, 'Path B total': tb,
                 'Path B %': f'{cb/tb:.0%}' if tb else '\u2013'})
roll = pd.DataFrame(rows)
sum_row = {'sample': 'ALL 10 (this is the confusion matrix total)',
           'Path A correct': roll['Path A correct'].sum(), 'Path A total': roll['Path A total'].sum(),
           'Path A %': f"{roll['Path A correct'].sum() / roll['Path A total'].sum():.0%}",
           'Path B correct': roll['Path B correct'].sum(), 'Path B total': roll['Path B total'].sum(),
           'Path B %': f"{roll['Path B correct'].sum() / roll['Path B total'].sum():.0%}"}
roll = pd.concat([roll, pd.DataFrame([sum_row])], ignore_index=True)

def highlight_low(row):
    if row['sample'].startswith('ALL'):
        return ['font-weight: bold; border-top: 2px solid black'] * len(row)
    try:
        pct = int(row['Path A %'].rstrip('%'))
    except ValueError:
        pct = 100
    return ['background-color: #fdecea' if pct < 50 else ''] * len(row)

display(roll.style.apply(highlight_low, axis=1).hide(axis='index'))


sample,Path A correct,Path A total,Path A %,Path B correct,Path B total,Path B %
sample1,27,31,87%,30,34,88%
sample2,18,28,64%,20,32,62%
sample3,13,15,87%,16,20,80%
sample4,2,2,100%,2,4,50%
sample5,12,17,71%,17,23,74%
sample6,1,12,8%,1,19,5%
sample7,2,2,100%,2,4,50%
sample8,13,13,100%,19,19,100%
sample9,11,11,100%,16,16,100%
sample10,13,13,100%,19,19,100%


## Takeaways

- Most documents match cleanly, or nearly so — the model correctly finds
  titles, section headings, questions and answers in the great majority of
  cases.
- Where it goes wrong, the mistake is almost always about **where one piece
  of text ends and the next begins**, not about misunderstanding the content.
  Two patterns show up repeatedly:
  - **Gluing a heading onto its answer** instead of keeping them as two
    separate items (this is what happens in the document with the lowest
    score, above).
  - **Splitting one long answer into pieces**, where a leftover sentence
    fragment gets mistaken for a new question.
- Because these are structural mistakes rather than misunderstandings, a small
  number of poorly-scoring documents can pull the overall average down a lot
  — worth knowing before reading a single "average score" for this model as
  the whole story.
